# CHSA Medical Triage Agent - Kaggle Model Comparison

Compare base, SFT, and DPO serving behavior with vLLM structured JSON constraints before measuring calibration metrics.

## Runtime checks

Enable a Kaggle GPU accelerator before running vLLM.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

print("python=", sys.version)
print("cwd=", Path.cwd())
subprocess.run(["nvidia-smi"], check=False)

## Clone or update repository

In [ ]:
REPO_URL = "https://github.com/Nhkp/medical-triage-agent.git"
REPO_DIR = Path("/kaggle/working/medical-triage-agent")

if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists():
        raise RuntimeError(f"{REPO_DIR} exists but is not a git checkout")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
print("repo=", Path.cwd())

## Install serving and evaluation dependencies

Do not reinstall Torch; Kaggle owns the CUDA runtime.

In [ ]:
!python -m pip install -q -U fastapi uvicorn vllm huggingface_hub pandas

## Load Hugging Face credentials

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient

    os.environ.setdefault("HF_TOKEN", UserSecretsClient().get_secret("HF_TOKEN"))
except (ImportError, KeyError, OSError, RuntimeError) as exc:
    print("Kaggle secret lookup skipped:", exc)

if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("Set HF_TOKEN as a Kaggle Secret or environment variable")

subprocess.run(["hf", "auth", "whoami"], check=False)

## Dry run

Verify model IDs, adapter IDs, dataset path, and output paths without loading model weights.

In [ ]:
!python scripts/evaluate_model_comparison.py --dry-run

## Sequential vLLM + FastAPI comparison

Each run starts one model configuration, evaluates it, then stops the server before moving to the next adapter.

In [ ]:
import signal
import time
from urllib.request import urlopen

BASE_MODEL = "Qwen/Qwen3-1.7B-Base"
MODELS = {
    "base": None,
    "sft": "Lokhidor/medical-triage-qwen3-sft-lora",
    "dpo": "Lokhidor/medical-triage-qwen3-dpo-lora",
}


def wait_for(url, name, timeout=900):
    deadline = time.time() + timeout
    last_error = ""
    while time.time() < deadline:
        try:
            with urlopen(url, timeout=5) as response:
                if 200 <= response.status < 500:
                    print(name, "ready:", url)
                    return
        except (OSError, TimeoutError) as exc:
            last_error = str(exc)
        time.sleep(2)
    raise TimeoutError(f"{name} not ready: {last_error}")


def stop(process):
    if process.poll() is None:
        process.send_signal(signal.SIGTERM)
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.kill()


for name, adapter in MODELS.items():
    vllm_cmd = [
        sys.executable,
        "-m",
        "vllm.entrypoints.openai.api_server",
        "--model",
        BASE_MODEL,
        "--host",
        "127.0.0.1",
        "--port",
        "8000",
        "--gpu-memory-utilization",
        "0.70",
        "--max-model-len",
        "4096",
        "--enforce-eager",
    ]
    model_name = BASE_MODEL
    if adapter:
        model_name = f"medical-triage-{name}"
        vllm_cmd += ["--enable-lora", "--lora-modules", f"{model_name}={adapter}"]

    env = os.environ.copy()
    env["PYTHONPATH"] = str(REPO_DIR / "src")
    env["VLLM_BASE_URL"] = "http://127.0.0.1:8000/v1"
    env["VLLM_MODEL_ID"] = model_name

    print("starting", name, adapter or BASE_MODEL)
    vllm = subprocess.Popen(vllm_cmd, env=env, text=True)
    api = None
    try:
        wait_for("http://127.0.0.1:8000/v1/models", "vLLM")
        api = subprocess.Popen(
            [
                sys.executable,
                "-m",
                "uvicorn",
                "medical_triage_agent.api:create_app",
                "--factory",
                "--host",
                "127.0.0.1",
                "--port",
                "8080",
            ],
            env=env,
            text=True,
        )
        wait_for("http://127.0.0.1:8080/health", "FastAPI")
        subprocess.run(
            [
                sys.executable,
                "scripts/evaluate_model_comparison.py",
                "--models",
                name,
                "--url",
                "http://127.0.0.1:8080",
                "--output-dir",
                "outputs/evaluations",
            ],
            check=True,
        )
    finally:
        if api:
            stop(api)
        stop(vllm)

## Display and export results

In [ ]:
import json

import pandas as pd

rows = []
for path in sorted(Path("outputs/evaluations").glob("model_comparison_*.json")):
    payload = json.loads(path.read_text(encoding="utf-8"))
    rows.append({"model": payload["model"], **payload["metrics"]})
summary = pd.DataFrame(rows)
summary.to_csv("outputs/evaluations/model_comparison_summary.csv", index=False)
display(summary)
print("Expected files:")
print("outputs/evaluations/model_comparison_base.json")
print("outputs/evaluations/model_comparison_sft.json")
print("outputs/evaluations/model_comparison_dpo.json")
print("outputs/evaluations/model_comparison_summary.csv")
!zip -r outputs-evaluations-model-comparison.zip outputs/evaluations/model_comparison_*.json outputs/evaluations/model_comparison_summary.csv
print("Download outputs-evaluations-model-comparison.zip from the Kaggle output panel.")